In [1]:
!pip install -q langchain langchain-community transformers accelerate faiss-cpu sentence-transformers

#Load Hugging Face Model

In [5]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model="gpt2",   # lightweight model
    max_new_tokens=100
)

llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#Basic LLM Call

In [6]:
response = llm.invoke("Explain Artificial Intelligence in simple terms")
print(response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain Artificial Intelligence in simple terms.

How do we create artificial intelligence in a way that is easy to understand and easily understood?

It's a lot harder. The AI is not very simple to understand, but the concepts are there. It's very difficult to learn. It's time consuming.

How do we automate things to make them more efficient?

We can automate what we are doing. We can automate things so that you can do things a little bit faster. We can automate things so that


#Prompt Template Example

In [9]:
from langchain_core.prompts import PromptTemplate
template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms for beginners"
)

prompt = template.format(topic="Machine Learning")

response = llm.invoke(prompt)
print(response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain Machine Learning in simple terms for beginners, and learn to use the concepts of data mining and machine learning as a way to build your own applications in a very short amount of time.

This class is designed to introduce you to data mining and machine learning by presenting you with the basics of machine learning, along with a video introduction to Machine Learning Basics. The classes are divided into three chapters:

Machine Learning Basics. This class is a great way to get into the fundamentals of machine learning, and will be a great introduction to


#Simple Chain

In [10]:
from langchain_core.prompts import PromptTemplate

# Step 1: Create prompt
template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms"
)

# Step 2: Create chain (NEW WAY)
chain = template | llm

# Step 3: Run
response = chain.invoke({"topic": "Deep Learning"})
print(response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain Deep Learning in simple terms, but also in the context of deep learning.

A key point to remember is that Deep Learning can be used to design neural networks. We can use simple techniques to build networks that learn the same thing over and over again.

We can also use deep learning to find a goal for the problem in the next iteration. This is a very simple approach to learning neural networks and how it's designed.

But most importantly, deep learning can be used to design new tools for solving


#Memory

In [13]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
template = PromptTemplate(
    input_variables=["input"],
    template="You are a helpful assistant. Answer: {input}"
)

# Store chat history
store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history
)



response1 = chain_with_memory.invoke(
    {"input": "Hi, my name is Alex"},
    config={"configurable": {"session_id": "1"}}
)

response2 = chain_with_memory.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "1"}}
)

print(response1)
print(response2)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain [HumanMessage(content='Hi, my name is Alex', additional_kwargs={}, response_metadata={})] in simple terms. That's a lot of work, but I don't think it's the most important part, and I'm not sure it matters much to you. I understand the technical aspects of the question, and I'm sure you understand how it would be handled.

In the end, I think that these two options are quite similar, and I think it's one of the most important points in our development environment.

You get the idea: you can do it with a little more abstraction
Explain [HumanMessage(content='Hi, my name is Alex', additional_kwargs={}, response_metadata={}), AIMessage(content="Explain [HumanMessage(content='Hi, my name is Alex', additional_kwargs={}, response_metadata={})] in simple terms. That's a lot of work, but I don't think it's the most important part, and I'm not sure it matters much to you. I understand the technical aspects of the question, and I'm sure you understand how it would be handled.\n\nIn the en

#Agent with Tool

In [19]:
!pip install -q langchain-experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 6.8 MB/s eta 0:00:00


In [24]:
from langchain.tools import tool

# Tool
@tool
def calculator(expression: str) -> str:
    """Performs math calculations"""
    try:
        return str(eval(expression))
    except:
        return "Error"

# Step 1: Ask LLM what to do
question = "What is 15 * 6?"

prompt = f"""
You are an AI agent.

If the question requires calculation, extract the math expression.

Question: {question}
"""

response = llm.invoke(prompt)

print("LLM reasoning:")
print(response)

# Step 2: Use tool manually
result = calculator.invoke("15 * 6")


print("\nTool Output:")
print(result)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LLM reasoning:

You are an AI agent.

If the question requires calculation, extract the math expression.

Question: What is 15 * 6?

Answer: 15 = 2.23

Question: What is 8 * 8?

Answer: 8 = 3.25

Question: What is 3 * 2?

Answer: 2 = 1.29

Question: What is 1 * 2?

Answer: 1 = 2.99

Question: What is 4 * 1?

Answer: 4 = 3.13

Question: What is 3 * 2?

Answer: 3 =

Tool Output:
90


#Document Loader

In [25]:
from langchain_community.document_loaders import TextLoader

# Create file
with open("data.txt", "w") as f:
    f.write("LangChain helps build LLM-powered applications.")

# Load document
loader = TextLoader("data.txt")
documents = loader.load()

print(documents)

[Document(metadata={'source': 'data.txt'}, page_content='LangChain helps build LLM-powered applications.')]


#Vector Store + Embeddings

In [26]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Create embeddings
embeddings = HuggingFaceEmbeddings()

# Store documents
vectorstore = FAISS.from_documents(documents, embeddings)

# Query
query = "What is LangChain?"
docs = vectorstore.similarity_search(query)

print(docs[0].page_content)

/tmp/ipykernel_3949/1253816419.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/tmp/ipykernel_3949/1253816419.py:5: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

LangChain helps build LLM-powered applications.


#FINAL INTEGRATED PIPELINE

In [27]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    input_variables=["input"],
    template="Answer clearly: {input}"
)

chain = template | llm

print(chain.invoke({"input": "What is Data Science?"}))
print(chain.invoke({"input": "Explain in one line"}))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer clearly: What is Data Science?

Data Science is a group of scientists who study the nature of data. Data scientist are known for their ability to understand the flow of information, to understand how data is distributed, to understand how data is used, and to understand data's dynamics. Their main focus is to understand how data is used, analyzed, and analyzed. To that end, they work on a single problem with specific focus: how to create datasets.

The primary method they use for the creation of data is the
Answer clearly: Explain in one line that the "The American Way" is a way to build a better society.

The American Way is a way to build a better society by building a better society.

In a nutshell: Yes, we need to build a better society.

But, we cannot get there by constructing a better society.

You're going to be building a better world by building a better world.

You're going to be building a better world by building a better world.



In [29]:
import json

with open("Gen_ai_task_02.ipynb", "r") as f:
    data = json.load(f)

if "widgets" in data.get("metadata", {}):
    del data["metadata"]["widgets"]

with open("Gen_ai_task_02.ipynb", "w") as f:
    json.dump(data, f)

print("Clean notebook saved!")

FileNotFoundError: [Errno 2] No such file or directory: 'Gen_ai_task_02.ipynb'